# 05. Inspect embedding outputs (AD cohort)

快速查看 `embedding_topN.parquet`、`cohort_labels`、`feature_correlations.xlsx`，
并确认 embedding 的 row 和标签表对齐。路径不对就改 `EMB_DIR`。


In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

EMB_DIR = Path('output/embeddings')

EMBEDDINGS = {f'top{n}': EMB_DIR / f'embedding_top{n}.parquet' for n in [10, 20, 30]}
LABELS_PATH = EMB_DIR / 'cohort_labels.parquet'
CORR_XLSX = EMB_DIR / 'feature_correlations.xlsx'

print('EMB_DIR:', EMB_DIR.resolve())
for name, p in {**EMBEDDINGS, 'labels': LABELS_PATH, 'corr_xlsx': CORR_XLSX}.items():
    print(f'{name:10s} exists = {p.exists()}  {p}')


## 1. Embedding schema (shape + column names)

In [ ]:
for name, p in EMBEDDINGS.items():
    if not p.exists():
        print(f'[{name}] not found'); continue
    pf = pq.ParquetFile(p)
    print('=' * 70)
    print(name, '->', p.name)
    print('rows:', pf.metadata.num_rows, '| columns:', len(pf.schema.names))
    print('first columns:', pf.schema.names[:8])


## 2. Embedding heads

In [ ]:
for name, p in EMBEDDINGS.items():
    if p.exists():
        df = pd.read_parquet(p)
        print('=' * 70)
        print(name, 'shape:', df.shape)
        display(df.head())


## 3. Label table + alignment check

In [ ]:
labels = pd.read_parquet(LABELS_PATH) if LABELS_PATH.exists() else None
if labels is not None:
    print('labels shape:', labels.shape)
    display(labels.head())
    print('label counts:')
    print(labels['label'].value_counts())
    if EMBEDDINGS['top10'].exists():
        top10 = pd.read_parquet(EMBEDDINGS['top10'], columns=['row_id'])
        aligned = top10['row_id'].reset_index(drop=True).equals(
            labels['row_id'].reset_index(drop=True))
        print('row_id aligned (top10 vs labels):', aligned)
else:
    print('cohort_labels.parquet not found; run notebook 04 first.')


## 4. Top correlated features

In [ ]:
if CORR_XLSX.exists():
    xls = pd.ExcelFile(CORR_XLSX)
    print('Sheets:', xls.sheet_names)
    top = pd.read_excel(CORR_XLSX, sheet_name=xls.sheet_names[0])
    print('Top selected features by |Pearson r|:')
    display(top.head(30))
else:
    print('feature_correlations.xlsx not found; run notebook 03 first.')
